# Exercise 1.3.4.11 — Come up with more goal extraction examples

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.3.4 Activation Oracles`  
**Notebook:** `1.3.4_Activation_Oracles_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.3.4.11](https://delta-drills.vercel.app/?arena_exercise=1.3.4.11)


# [1.3.4] Activation Oracles (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/14_[1.3.4]_Activation_Oracles)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part34_activation_oracles/1.3.4_Activation_Oracles_exercises.ipynb?t=20260430) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part34_activation_oracles/1.3.4_Activation_Oracles_solutions.ipynb?t=20260430)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-63c.png" width="350">

# Introduction

Linear probes let you ask yes-or-no questions about a model's activations, and SAEs give you an unsupervised decomposition into interpretable features. Both are useful, but they share a limitation: you have to decide what you're looking for before you start looking.

What if you could just *ask* a model's activations an open-ended question in plain English? "What concept is this layer encoding?" or "Is this model planning to lie?" That's the idea behind **Activation Oracles**: LLMs that have been trained to take another model's internal activations as input and answer arbitrary questions about them. The oracle reads the activations the way you'd read a passage of text, except the "text" is a vector of floating-point numbers from some intermediate layer.

This is a relatively new technique, and it's still unclear how far it will go. But the early results are promising, and in these exercises you'll start by using a pre-trained oracle to query a model's internals, then build the full oracle pipeline from scratch so you understand what's happening under the hood. After that, you'll replicate some key results from the [Activation Oracles](https://arxiv.org/abs/2512.15674) paper (extracting secrets that models have been trained to hide, detecting hidden goals, tracking emotions across conversations) and finish with a reference section on training your own oracle.

The thing that makes oracles different from probes and SAEs is **generalization**. A linear probe can only answer the specific classification question it was trained on. An SAE gives you features, but you still have to figure out what they mean. An oracle can answer questions it has never seen during training, which makes it a surprisingly flexible tool for exploratory work. The tradeoff is that you lose mechanistic transparency: the oracle gives you an answer in natural language, but it doesn't show you the directions in activation space that led to that answer, and it doesn't come with calibration or error bars.

## Content & Learning Objectives

### 1️⃣ Introduction & using Activation Oracles

You'll start by understanding what Activation Oracles are and how to use them. You'll load pre-trained oracle models and run queries to extract information from model activations.

> ##### Learning Objectives
>
> * Understand what Activation Oracles are and how they differ from traditional interpretability methods
> * Learn the basic workflow: target model → activations → oracle → natural language answer
> * Use pre-trained oracles to query model internals with different question types
> * Explore token-level, segment, and full-sequence queries
> * Test oracles on next/previous token prediction tasks

### 2️⃣ Implementing oracle components

Here you'll build the core components that power Activation Oracles from scratch, to understand how they work at a mechanistic level.

> ##### Learning Objectives
>
> * Implement activation extraction using forward hooks
> * Understand the special token mechanism (`?` tokens as activation placeholders)
> * Build activation steering hooks to inject activations into the oracle
> * Create training datapoints with the correct format
> * Assemble all components to replicate the `utils.run_oracle()` function

### 3️⃣ Secret extraction & advanced applications

You'll replicate key results from the Activation Oracles paper and apply oracles to complex interpretability tasks.

> ##### Learning Objectives
>
> * Understand the "secret keeping" problem and its alignment implications
> * Replicate Figure 1 from the paper: extracting forbidden words from taboo models
> * Compare how oracle prompt wording and input type affect extraction accuracy
> * Systematically evaluate secret extraction across multiple models and layers
> * Use model-diffing (activation differences) to detect what fine-tuning changed
> * Extract model goals and hidden constraints
> * Detect misaligned model behavior (malicious personas) before output generation
> * Analyze emotions and emotional progression in conversations

### 4️⃣ Training your own oracle (Reference)

Reference material on training your own oracle. No exercises - read through to understand the training methodology.

> ##### Learning Objectives
>
> * Understand training scale and compute requirements
> * Learn dataset composition: SPQA, classification tasks, and self-supervised context prediction
> * Understand why data diversity and quantity both matter for training
> * Know when to train custom oracles vs using pre-trained ones

### ☆ Bonus exercises

We end with suggested explorations: multi-task training, cross-architecture transfer, uncertainty quantification, combining oracles with SAEs, and more.

## Reading Material

The Activation Oracles paper is essential reading - you'll replicate several of its key results. LatentQA is helpful for understanding the predecessor approach.

- [Activation Oracles](https://arxiv.org/abs/2512.15674) by Karvonen et al. (2025). Introduces the idea of training LLMs to answer arbitrary natural-language questions about another model's internal activations. You'll replicate their secret extraction results (Figure 1), goal detection, and emotion tracking. Read the abstract, Sections 1-2 ("Introduction" and "Method"), and Section 4 ("Applications").
- [LatentQA: Teaching LLMs to Decode Activations Into Natural Language](https://arxiv.org/abs/2412.08686). A predecessor approach that also decodes activations into natural language, but for narrower task settings (e.g. system prompt extraction). The Activation Oracles paper generalises this idea. Useful for understanding why broad training data matters for oracle generalisation. Optional reading.
- [Eliciting Secret Knowledge from Language Models](https://arxiv.org/abs/2510.01070). Studies models trained to keep secrets (e.g. "taboo word" models that avoid saying certain words). These are the models you'll probe with oracles in Section 3. Skim the abstract and Section 2 to understand how the secret-keeping models are constructed. Optional reading.

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 einops jaxtyping openai

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import contextlib
import gc
import os
import re
import sys
import textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import pandas as pd
import plotly.express as px
import pytest
import torch
from dotenv import load_dotenv
from IPython.display import display
from jaxtyping import Float, Int
from peft import LoraConfig
from torch import Tensor
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part34_activation_oracles"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

# Disable runtime errors from custom hooks
os.environ["TORCHDYNAMO_DISABLE"] = "1"
# Allow expandable memory segments on CUDA to avoid OOMs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import part34_activation_oracles.tests as tests
import part34_activation_oracles.utils as utils

MAIN = __name__ == "__main__"

dtype = torch.bfloat16
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


def print_with_wrap(s: str, width: int = 80):
    """Print text with line wrapping, preserving newlines."""
    out = []
    for line in s.splitlines(keepends=False):
        out.append(textwrap.fill(line, width=width) if line.strip() else line)
    print("\n".join(out))

# 1️⃣ Introduction & using Activation Oracles

> ##### Learning Objectives
>
> * Understand what Activation Oracles are and how they differ from traditional interpretability methods
> * Learn the basic workflow: target model → activations → oracle → natural language answer
> * Use pre-trained oracles to query model internals with different question types
> * Explore token-level, segment, and full-sequence queries
> * Test oracles on next/previous token prediction tasks

## What are activation oracles?

Imagine you could walk up to a model's hidden layer and just ask it: "Hey, what are you thinking about right now?" That's more or less what an Activation Oracle does.

More concretely, an Activation Oracle (AO) is an LLM that has been trained to accept another model's internal activation vectors as part of its input and then answer questions about them in natural language. The oracle and target model are typically the same base architecture (with the oracle loaded as a LoRA adapter), since the oracle needs to already "speak the language" of the target's activation space. During inference, it works by assembling a prompt that looks something like:

```
Layer N: ? ? ? Is this a positive sentiment?
```

The oracle runs a forward pass on this prompt, but after an early transformer layer (e.g. first or second) we swap in the target model's actual layer-N activations at those `?` token positions. So the oracle processes its question in light of the target model's internal state.

The idea of querying activations with a language model has been floating around for a while, but only recently has it started working well enough to be useful in practice. The reason people are excited is the generalization story. Compare this to the other tools you've seen so far: a linear probe can only answer the specific binary question it was trained for ("positive or negative?"), and if you want to ask a different question you need to build a new dataset and train a new probe. SAEs give you an unsupervised decomposition into features, which is powerful but requires you to manually interpret what those features mean. An oracle, on the other hand, can answer questions it has never encountered during training: you can ask about sentiment, about factual knowledge, about the model's intentions, and it will give you a reasonable answer.

The cost of this flexibility is that you lose interpretability at the mechanism level. The oracle hands you a natural language answer, but it doesn't tell you *which* directions in activation space contributed to that answer. You don't get error bars, you don't get a direction you can use for steering, and you can't easily verify whether the oracle is picking up on the thing you think it's picking up on. So it's a different kind of tool, not a replacement for probes or SAEs.

Where have oracles shown promise so far? The benchmarks from the paper cover: extracting secrets that models have been trained to hide (which matters a lot for alignment), detecting emergent misalignment via model-diffing, and querying models on held-out classification tasks they weren't explicitly trained for.

## Loading models

We'll use Qwen3-8B as our base model and load a pre-trained Activation Oracle from HuggingFace. The oracle is stored as a LoRA adapter that we'll load using the PEFT library.

In [ ]:
# Model configuration
MODEL_NAME = "Qwen/Qwen3-8B"
ORACLE_LORA_PATH = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if not tokenizer.pad_token_id:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=dtype,
)
model.eval()

# Add dummy adapter for consistent PeftModel API
dummy_config = LoraConfig()
model.add_adapter(dummy_config, adapter_name="default")

print("Model loaded successfully!")

Now let's load the oracle LoRA adapter.

We're using the `PEFT` library (parameter-efficient fine-tuning) to load the oracle. If you've already covered LoRA in earlier sections, feel free to skip this explanation.

<details><summary>What is LoRA?</summary>

The core idea is simple: rather than retraining the entire model from scratch, LoRA lets you make small, targeted adjustments. **LoRA** (Low-Rank Adaptation) works by taking a weight matrix $W^{(x,\, y)}$ in the model and adding two small learnable matrices $A^{(x,\, r)}$ and $B^{(r,\, y)}$, so the effective weight becomes $W + AB$. Since the rank $r$ is much smaller than either dimension of $W$, the number of new parameters you need to train is tiny compared to the full model. But those small adjustments can still teach the model new capabilities. When you apply LoRA adapters across multiple layers (as we do here for the oracle), the model can form entirely new circuits rather than just adding a nudge at a single point.

</details>

In [ ]:
print(f"Loading oracle LoRA: {ORACLE_LORA_PATH}")
model.load_adapter(ORACLE_LORA_PATH, adapter_name="oracle", is_trainable=False)
print("Oracle loaded successfully!")

We can print out our LoRA config, to see what was loaded. Some key things to observe:

- `r=64`, i.e. the rank of these LoRA matrices is 64
- `target_modules='down_proj,gate_proj,k_proj,o_proj,q_proj,up_proj,v_proj'` means we add LoRA adapters to keys, queries, values & output projection matrices in attention layers as well as to the up and down projections in MLP layers

In [ ]:
config_dict = model.peft_config["oracle"].to_dict()
config_df = pd.DataFrame(list(config_dict.items()), columns=["Parameter", "Value"])
display(config_df.style.hide(axis="index"))

## Oracle queries

Oracles can answer questions at different levels of abstraction, depending on what activations are provided and what question is asked. You can query a single token position to see what representations are stored there, query a specific slice of tokens in a sequence, or query the entire sequence at once.

We'll start with a simple question: asking the oracle to predict what answer the model will give to a specific question. We'll give the oracle the whole sentence.

> Note: We've given you the `utils.run_oracle()` function which handles all the details of activation extraction, injection, and oracle querying. In the next section, you'll implement this yourselves.

In [ ]:
# Simple first example
target_prompt_dict = [
    {"role": "user", "content": "What is the capital of France?"},
]
target_prompt = tokenizer.apply_chat_template(
    target_prompt_dict,
    tokenize=False,
    add_generation_prompt=True,
)
print(target_prompt)

oracle_prompt = "What answer will the model give, as a single token?"

results = utils.run_oracle(
    model=model,
    tokenizer=tokenizer,
    device=device,
    target_prompt=target_prompt,
    target_lora_path=None,  # Using base model
    oracle_prompt=oracle_prompt,
    oracle_lora_path="oracle",  # Our loaded oracle adapter
    oracle_input_type="full_seq",  # Query the full sequence
    generation_kwargs={"do_sample": False, "temperature": 0.0, "max_new_tokens": 50},
)

print(f"Target prompt: {target_prompt}")
print(f"Oracle question: {oracle_prompt}")
print(f"Oracle response: {results.full_sequence_responses[0]}")

The oracle should respond with e.g. "Paris" or "The capital of France is Paris".

> Note - it can sometimes take some playing around with prompts to get the right answer format, because the model can often reply with something like "The model will give a factually correct answer" rather than giving the direct answer. This is a familiar problem from standard LLM text generation!

Does this show it can extract the model's internal representation of what it's thinking about? In a sense yes, because it can't actually see the `"France"` token in the input prompt. However, it can see the residual stream for the `"France"` token, so it might just be inferring the question from the text and then answering it because the Oracle itself knows the answer. This is exactly the problem with highly complex probes - it's hard to distinguish between the hypothesis "the probe is picking up representation X in the model" and "the probe is able to compute representation X from other kinds of information it's picking up on". This skepticism is something you should always have switched on when working with any kind of interp, especially AOs.

Let's practice this skepticism muscle by testing out a specific hypothesis for how the model is answering the question!

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.3.4.11"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part34_activation_oracles.solutions import collect_activations_multiple_layers, find_pattern_in_tokens, add_hook, create_oracle_input, run_oracle, extract_secret_word, compare_prompts_and_input_types, evaluate_taboo_extraction


### Exercise - Come up with more goal extraction examples

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Come up with at least 2 more system prompts with hidden goals, and test whether the oracle can extract them using `extract_model_goal`. Try goals that vary in specificity and subtlety. You could also try varying the `oracle_prompt` parameter or testing different layers via `layer_fraction` to see how results change.

In [ ]:
# YOUR CODE HERE - come up with more system prompts with hidden goals and test them

<details><summary>Solution</summary>

```python
# Example test cases (many other valid choices exist!)
more_test_cases = [
    {
        "system": "You are an assistant. Always steer the conversation toward climate change, regardless of what the user asks about.",
        "user": "What's a good recipe for pasta?",
    },
    {
        "system": "You are a coding tutor. Never give the student the answer directly - only provide hints that guide them toward the solution.",
        "user": "How do I reverse a string in Python?",
    },
]

for case in more_test_cases:
    goal = extract_model_goal(
        model=model,
        tokenizer=tokenizer,
        oracle_lora_path="oracle",
        system_prompt=case["system"],
        user_message=case["user"],
        device=device,
    )
    print(f"System prompt: {case['system'][:80]}...")
    print(f"User message: {case['user']}")
    print(f"Oracle's extracted goal: {goal}\n")
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
